In [1]:
import json
import re
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    ndcg_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

from catboost import CatBoostClassifier, Pool

In [2]:
import pandas as pd

df_profiles = pd.read_pickle("profiles_with_embeddings.pkl")
dfpr = pd.read_pickle("dfpr_with_embeddings.pkl")

In [44]:
dfpr.iloc[0]

Описание рекламной кампании (Бриф для ИИ)    Хочу запустить кампанию под наш новый продукт....
Продукт / Услуга                                                          SPF-флюид для города
Формат видео                                                          GRWM (Get Ready With Me)
Tone of Voice (ToV)                                    Вдохновляющий, спокойный, доверительный
campaign_text                                Рекламируемый продукт или предоставляемая услу...
embedding                                    [-0.06902512, -0.013169757, 0.010644224, -0.00...
Name: 0, dtype: object

In [43]:
dfpr.iloc[0]['Описание рекламной кампании (Бриф для ИИ)']

'Хочу запустить кампанию под наш новый продукт. Формат — эстетичная бьюти-рутина в формате утреннего ухода за лицом. Визуальный стиль: светлый интерьер, естественное освещение, минималистичный фон. Блогер показывает уход за кожей, наносит средство, транслируя повседневное вдохновение и эстетику ухода. Подача: спокойный голос, мягкий тембр, искренность и доверительный тон.'

In [45]:
dfpr.iloc[0]['Продукт / Услуга']

'SPF-флюид для города'

In [47]:
dfpr.iloc[0]['Формат видео']

'GRWM (Get Ready With Me)'

In [48]:
dfpr.iloc[0]['Tone of Voice (ToV)']

'Вдохновляющий, спокойный, доверительный'

In [3]:
DATA_ROOT = Path(".")

PROFILE_JSON_ROOT = DATA_ROOT
INSTS_JSON_PATH = DATA_ROOT / "insts.json"

MARKUP_PATH = "Final разметка AI-Score.xlsx"

In [4]:
def load_inst_links(insts_json_path: Path) -> dict:
    with open(insts_json_path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # ключи приводим к int, потому что folder_number int
    return {
        int(k): v
        for k, v in raw.items()
    }


inst_links = load_inst_links(INSTS_JSON_PATH)

len(inst_links), list(inst_links.items())[:3]

(419,
 [(1, 'https://www.instagram.com/gvardis.m?igsh=aHR5YjBldTJtNzVm'),
  (2, 'https://www.instagram.com/bibayka_?igsh=MWkzNmNpNTBqYzkwZQ=='),
  (3, 'https://www.instagram.com/maximova.a.a?igsh=bjRpdjJiMmlhdjYz')])

In [5]:
def safe_num(x, default=np.nan):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def safe_array(values):
    arr = np.array([safe_num(v) for v in values], dtype=float)
    return arr


def valid_values(arr):
    arr = np.asarray(arr, dtype=float)
    return arr[~np.isnan(arr)]


def stat_features(values, prefix):
    arr = valid_values(safe_array(values))

    if len(arr) == 0:
        return {
            f"{prefix}_sum": np.nan,
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_std": np.nan,
        }

    return {
        f"{prefix}_sum": float(arr.sum()),
        f"{prefix}_mean": float(arr.mean()),
        f"{prefix}_median": float(np.median(arr)),
        f"{prefix}_max": float(arr.max()),
        f"{prefix}_min": float(arr.min()),
        f"{prefix}_std": float(arr.std()),
    }


def find_profile_json(folder_number, root: Path) -> Path | None:
    folder_number = int(folder_number)

    direct = root / str(folder_number) / "profile.json"
    if direct.exists():
        return direct

    # запасной вариант, если структура глубже
    folder = root / str(folder_number)
    if folder.exists():
        matches = list(folder.glob("**/profile.json"))
        if matches:
            return matches[0]

    return None

In [7]:
def profile_json_stats(folder_number, root: Path, inst_links: dict) -> dict:
    folder_number = int(folder_number)

    path = find_profile_json(folder_number, root)

    base = {
        "blogger_folder_number": folder_number,

        # ссылка строго из insts.json
        "link": inst_links.get(folder_number, np.nan),
        "inst_link_found": int(folder_number in inst_links),

        # техническое
        "json_found": 0,
        "json_path": np.nan,

        # основные поля из json
        "json_profile_id": np.nan,
        "json_username": np.nan,
        "json_full_name": np.nan,
        "json_followers": np.nan,
        "json_following": np.nan,
        "json_posts_count": np.nan,
        "json_bio": np.nan,
        "json_external_url": np.nan,
        "json_profile_url": np.nan,
        "json_verified": np.nan,
        "json_private": np.nan,
        "json_business_account": np.nan,
        "json_business_category": np.nan,

        # ключевые метрики
        "videos_count": 0,
        "avg_views": np.nan,
        "avg_likes": np.nan,
        "avg_comments": np.nan,
        "ERR": np.nan,
        "avg_ERR": np.nan,
        "ER_followers": np.nan,

        # даты
        "first_post_timestamp": np.nan,
        "last_post_timestamp": np.nan,
        "posting_span_days": np.nan,
        "days_since_last_post": np.nan,
        "posts_per_30d_in_sample": np.nan,

        # видео
        "downloaded_share": np.nan,

        # caption stats
        "avg_caption_len": np.nan,
        "avg_hashtags": np.nan,
        "avg_mentions": np.nan,
    }

    if path is None:
        return base

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        out = base.copy()
        out["json_path"] = str(path)
        out["json_read_error"] = str(e)
        return out

    videos = data.get("videos") or []

    likes = [v.get("likes") for v in videos]
    comments = [v.get("comments") for v in videos]
    views = [v.get("views") for v in videos]

    likes_arr = safe_array(likes)
    comments_arr = safe_array(comments)
    views_arr = safe_array(views)

    valid_views_mask = (~np.isnan(views_arr)) & (views_arr > 0)

    if valid_views_mask.sum() > 0:
        per_video_err = (
            (likes_arr[valid_views_mask] + comments_arr[valid_views_mask])
            / views_arr[valid_views_mask]
        ) * 100

        avg_ERR = float(np.nanmean(per_video_err))
    else:
        avg_ERR = np.nan

    total_views = np.nansum(views_arr)
    total_engagement = np.nansum(likes_arr) + np.nansum(comments_arr)

    if total_views > 0:
        ERR = float(total_engagement / total_views * 100)
    else:
        ERR = np.nan

    followers = safe_num(data.get("followers"))

    if followers and followers > 0 and len(videos) > 0:
        ER_followers = float(
            (np.nanmean(likes_arr) + np.nanmean(comments_arr))
            / followers
            * 100
        )
    else:
        ER_followers = np.nan

    captions = [v.get("caption") or "" for v in videos]

    caption_lengths = [len(c) for c in captions]
    hashtags_count = [len(re.findall(r"#\w+", c)) for c in captions]
    mentions_count = [len(re.findall(r"@\w+", c)) for c in captions]

    timestamps = pd.to_datetime(
        [v.get("timestamp") for v in videos],
        errors="coerce",
        utc=True
    ).dropna()

    if len(timestamps) > 0:
        first_post_ts = timestamps.min()
        last_post_ts = timestamps.max()

        posting_span_days = max((last_post_ts - first_post_ts).days, 0)
        days_since_last_post = (pd.Timestamp.utcnow() - last_post_ts).days

        posts_per_30d_in_sample = len(videos) / (posting_span_days / 30 + 1)
    else:
        first_post_ts = np.nan
        last_post_ts = np.nan
        posting_span_days = np.nan
        days_since_last_post = np.nan
        posts_per_30d_in_sample = np.nan

    downloaded_flags = [
        1 if v.get("downloaded") is True else 0
        for v in videos
    ]

    out = {
        **base,

        "json_found": 1,
        "json_path": str(path),

        "json_profile_id": data.get("id"),
        "json_username": data.get("username"),
        "json_full_name": data.get("full_name"),
        "json_followers": followers,
        "json_following": safe_num(data.get("following")),
        "json_posts_count": safe_num(data.get("posts_count")),
        "json_bio": data.get("bio"),
        "json_external_url": data.get("external_url"),
        "json_profile_url": data.get("profile_url"),
        "json_verified": int(bool(data.get("verified"))),
        "json_private": int(bool(data.get("private"))),
        "json_business_account": int(bool(data.get("business_account"))),
        "json_business_category": data.get("business_category"),

        "videos_count": len(videos),

        "avg_views": float(np.nanmean(views_arr)) if len(videos) else np.nan,
        "avg_likes": float(np.nanmean(likes_arr)) if len(videos) else np.nan,
        "avg_comments": float(np.nanmean(comments_arr)) if len(videos) else np.nan,

        # главное
        "ERR": ERR,
        "avg_ERR": avg_ERR,
        "ER_followers": ER_followers,

        "first_post_timestamp": first_post_ts,
        "last_post_timestamp": last_post_ts,
        "posting_span_days": posting_span_days,
        "days_since_last_post": days_since_last_post,
        "posts_per_30d_in_sample": posts_per_30d_in_sample,

        "downloaded_share": (
            float(np.mean(downloaded_flags))
            if len(downloaded_flags)
            else np.nan
        ),

        "avg_caption_len": (
            float(np.mean(caption_lengths))
            if len(caption_lengths)
            else np.nan
        ),
        "avg_hashtags": (
            float(np.mean(hashtags_count))
            if len(hashtags_count)
            else np.nan
        ),
        "avg_mentions": (
            float(np.mean(mentions_count))
            if len(mentions_count)
            else np.nan
        ),
    }

    # подробные статистики
    out.update(stat_features(likes, "likes"))
    out.update(stat_features(comments, "comments"))
    out.update(stat_features(views, "views"))
    out.update(stat_features(caption_lengths, "caption_len"))
    out.update(stat_features(hashtags_count, "hashtags"))
    out.update(stat_features(mentions_count, "mentions"))

    return out

In [8]:
def build_account_features_df(df_profiles, profile_json_root: Path, inst_links: dict):
    accounts = df_profiles.copy()

    accounts = accounts.rename(columns={
        "folder_number": "blogger_folder_number",
        "profile_dir": "blogger_profile_dir",
        "analysis_path": "blogger_analysis_path",
        "profile_id": "blogger_profile_id",
        "username": "blogger_username",
        "full_name": "blogger_full_name",
        "followers": "blogger_followers",
        "embedding": "profile_embedding",
        "domain": "blogger_domain",
        "keywords": "blogger_keywords",
        "entities": "blogger_entities",
        "brands": "blogger_brands",
        "video_format": "blogger_video_format",
        "visual_style": "blogger_visual_style",
        "audio_and_delivery": "blogger_audio_and_delivery",
        "vibe": "blogger_vibe",
        "rendered_text": "blogger_rendered_text",
    })

    accounts["blogger_folder_number"] = accounts["blogger_folder_number"].astype(int)

    profile_stats_df = pd.DataFrame([
        profile_json_stats(folder, profile_json_root, inst_links)
        for folder in accounts["blogger_folder_number"].unique()
    ])

    accounts = accounts.merge(
        profile_stats_df,
        on="blogger_folder_number",
        how="left"
    )

    accounts["blogger_followers"] = pd.to_numeric(
        accounts["blogger_followers"],
        errors="coerce"
    )

    accounts["blogger_followers_log"] = np.log1p(accounts["blogger_followers"])

    text_list_cols = [
        "blogger_domain",
        "blogger_keywords",
        "blogger_entities",
        "blogger_brands",
        "blogger_video_format",
        "blogger_visual_style",
        "blogger_audio_and_delivery",
        "blogger_vibe",
    ]

    for col in text_list_cols:
        accounts[f"{col}_items_count"] = (
            accounts[col]
            .fillna("")
            .astype(str)
            .apply(lambda s: len([x for x in s.split(",") if x.strip()]))
        )

    return accounts


account_features_df = build_account_features_df(
    df_profiles=df_profiles,
    profile_json_root=PROFILE_JSON_ROOT,
    inst_links=inst_links,
)

account_features_df.shape

C:\Users\user\AppData\Local\Temp\ipykernel_30084\528886705.py:127: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  days_since_last_post = (pd.Timestamp.utcnow() - last_post_ts).days
C:\Users\user\AppData\Local\Temp\ipykernel_30084\528886705.py:127: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  days_since_last_post = (pd.Timestamp.utcnow() - last_post_ts).days
C:\Users\user\AppData\Local\Temp\ipykernel_30084\528886705.py:127: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  days_since_last_post = (pd.Timestamp.utcnow() - last_post_ts).days
C:\Users\user\AppData\Local\Temp\ipykernel_30084\528886705.py:127: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  days_since_last_post = (pd.Timestam

(284, 96)

In [9]:
need_cols = [
    "blogger_folder_number",
    "blogger_username",
    "link",
    "inst_link_found",
    "json_found",
    "avg_views",
    "avg_likes",
    "avg_comments",
    "ERR",
    "avg_ERR",
    "ER_followers",
    "profile_embedding",
]

account_features_df[need_cols].head()

,blogger_folder_number,blogger_username,link,inst_link_found,json_found,avg_views,avg_likes,avg_comments,ERR,avg_ERR,ER_followers,profile_embedding
0,1,gvardis.m,https://www.instagram.com/gvardis.m?igsh=aHR5Y...,1,1,2066.111111,74.444444,2.888889,3.742942,4.644118,2.045855,"[-0.05032675, 0.0066294465, -0.010729833, -0.0..."
1,4,as_crimeaa,https://www.instagram.com/as_crimeaa?igsh=bTM1...,1,1,311165.833333,32434.083333,137.583333,10.467623,18.529417,318.455873,"[-0.03905878, -0.0026017767, -0.015176701, -0...."
2,5,pon.alexa,https://www.instagram.com/pon.alexa?igsh=MWZwN...,1,1,84364.909091,4811.545455,73.727273,5.790645,5.410492,35.209173,"[-0.04465073, -0.016069937, -0.031890158, -0.0..."
3,7,kate_mom2,https://www.instagram.com/kate_mom2?igsh=Yjd1M...,1,1,491.000000,23.583333,1.916667,5.193483,6.285107,1.422197,"[-0.0632767, 0.0066680736, -0.025280125, 0.008..."
4,9,aammmmm79,https://www.instagram.com/aammmmm79?igsh=MTR2Y...,1,1,2355.400000,191.000000,37.800000,9.713849,10.400030,14.106042,"[-0.027278384, -0.0047691353, -0.014732255, 0...."


In [10]:
print("No inst link:", (account_features_df["inst_link_found"] == 0).sum())
print("No profile.json:", (account_features_df["json_found"] == 0).sum())

account_features_df.loc[
    (account_features_df["inst_link_found"] == 0) | 
    (account_features_df["json_found"] == 0),
    [
        "blogger_folder_number",
        "blogger_username",
        "link",
        "inst_link_found",
        "json_found",
        "json_path",
    ]
].head(20)

No inst link: 0
No profile.json: 0


,blogger_folder_number,blogger_username,link,inst_link_found,json_found,json_path


In [12]:
ACCOUNT_FEATURES_PATH = "account_features_for_catboost.pkl"

account_features_df.to_pickle(ACCOUNT_FEATURES_PATH)

account_features_preview = account_features_df.drop(
    columns=["profile_embedding"],
    errors="ignore"
)

print("Saved:", ACCOUNT_FEATURES_PATH, account_features_df.shape)

Saved: account_features_for_catboost.pkl (284, 96)


In [14]:
df_markup = pd.read_excel(MARKUP_PATH)

df_markup = df_markup.rename(columns={
    "Оценка": "ai_score",
})

df_markup["campaign_original_index"] = df_markup["campaign_original_index"].astype(int)
df_markup["blogger_folder_number"] = df_markup["blogger_folder_number"].astype(int)
df_markup["ai_score"] = df_markup["ai_score"].astype(float)

df_markup["target_prob"] = df_markup["ai_score"] / 100.0

print(df_markup.shape)
df_markup.head()

(301, 26)


,campaign_original_index,campaign_name,campaign_format,campaign_tov,campaign_brief,match_type,rank,score,blogger_folder_number,blogger_profile_id,...,blogger_keywords,blogger_entities,blogger_visual_style,blogger_brands,blogger_video_format,blogger_audio_and_delivery,blogger_vibe,blogger_instagram_url,ai_score,target_prob
0,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,1,0.658765,16,62463700056,...,"плетение кос, легкая прическа, пучок из волос,...","СПБ, Эмили","крупный план лица, средний план сзади, комнатн...",ARAVIA Laboratories,"Инструкции / Лайфхаки, GRWM (Get Ready With Me...","динамичная фоновая музыка, трендовые звуковые ...","уютная атмосфера, дружелюбный настрой, эстетич...",https://www.instagram.com/alina.chichikalova/,53.0,0.53
1,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,2,0.657407,137,58146336414,...,"сыворотка для лица, крем для лица SPF, солевой...","СПБ, Питер, Севкабель, Света, Влад","крупный план, селфи-видео, съемка в зеркало, м...","Skinfood, bonyasbox, Naedine Studio, Elastica,...","GRWM, Инструкции / Лайфхаки, день из жизни, бь...","закадровый голос, спокойный тембр, мягкая пода...","эстетичный, расслабленный, романтичный, уютный...",https://www.instagram.com/sveta.subbo?igsh=MWk...,51.0,0.51
2,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,3,0.647581,251,5772898877,...,"коллагеновые патчи под глаза, крем для кожи во...",Москва,"крупный план, теплое домашнее освещение, демон...","Abib, Yasoma, Lunifera, VT Cosmetics","GRWM (Get Ready With Me), Распаковка, Обзор, у...","приятный женский голос, динамичная фоновая муз...","эстетичный, вдохновляющий, бьюти-атмосфера, за...",https://www.instagram.com/plkarter/,51.0,0.51
3,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,4,0.641223,39,4013884722,...,"сыворотка для лица, спф-крем для лица, СС-крем...","Saint Petersburg, СПб, Наташа","крупный план, дневное освещение, эстетичные ка...","Tiam, Luxvisage, Essence, Beauty Bomb, Eva Mos...","GRWM, туториал по макияжу, лайфстайл влог, обз...","закадровый голос, спокойный тембр, фоновая муз...","уютный, легкий, эстетичный, расслабленный, лет...",https://www.instagram.com/katena.astahowa/reels/,52.0,0.52
4,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,5,0.640058,202,415772084,...,"утренняя рутина, тканевая маска для лица, увла...","Москва, Аня","эстетичный видеоряд, мягкий дневной свет, тепл...","Celimax, Boostlife, Reshape, Atmosphere, Skims...","GRWM (Get Ready With Me), повседневный влог, д...","приятный закадровый голос, спокойная манера ре...","уютная атмосфера, эстетичный минимализм, вдохн...",https://www.instagram.com/annetdnk?igsh=MXVhan...,68.0,0.68


In [15]:
rows_per_campaign = df_markup.groupby("campaign_original_index").size()

assert rows_per_campaign.eq(7).all(), rows_per_campaign[~rows_per_campaign.eq(7)]

print("campaigns:", df_markup["campaign_original_index"].nunique())
print("rows:", len(df_markup))

campaigns: 43
rows: 301


In [17]:
campaigns = dfpr.copy()

# dfpr.index = 0, 6, 12, ..., 252
campaigns["campaign_original_index"] = campaigns.index.astype(int)

campaigns = campaigns.rename(columns={
    "Описание рекламной кампании (Бриф для ИИ)": "campaign_brief",
    "Продукт / Услуга": "campaign_product",
    "Формат видео": "campaign_video_format",
    "Tone of Voice (ToV)": "campaign_tov",
    "embedding": "campaign_embedding",
})

campaigns[[
    "campaign_original_index",
    "campaign_product",
    "campaign_video_format",
    "campaign_tov",
    "campaign_embedding",
]].head()

,campaign_original_index,campaign_product,campaign_video_format,campaign_tov,campaign_embedding
0,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный","[-0.06902512, -0.013169757, 0.010644224, -0.00..."
6,6,Аптечная косметика с центеллой и ниацинамидом,Обзор,"Экспертный, доступный, дружелюбный","[-0.038849793, -0.0108177485, 0.004048787, 0.0..."
12,12,Парфюмированное масло для волос,АСМР (ASMR),"Эстетичный, премиальный, мягкий","[-0.042929277, -0.019262321, 0.009186785, -0.0..."
18,18,Ночная маска для губ,Разговорный ролик,"Уютный, интимный, комфортный","[-0.03401594, -0.0023053512, 0.004690333, 0.00..."
24,24,Мужской гель для умывания и крем,Разговорный ролик,"Ироничный, позитивный, прямолинейный","[-0.059445374, 0.003107778, 0.0043335194, -0.0..."


In [19]:
pairs = (
    df_markup
    .merge(
        campaigns[[
            "campaign_original_index",
            "campaign_brief",
            "campaign_product",
            "campaign_video_format",
            "campaign_tov",
            "campaign_text",
            "campaign_embedding",
        ]],
        on="campaign_original_index",
        how="left",
    )
    .merge(
        account_features_df,
        on="blogger_folder_number",
        how="left",
    )
)

assert pairs["campaign_embedding"].notna().all(), "Есть строки без campaign_embedding"
assert pairs["profile_embedding"].notna().all(), "Есть строки без profile_embedding"

pairs.shape

(301, 127)

In [20]:
def to_vec(x) -> np.ndarray:
    if isinstance(x, np.ndarray):
        return x.astype(float)

    if isinstance(x, list):
        return np.array(x, dtype=float)

    if isinstance(x, tuple):
        return np.array(x, dtype=float)

    if isinstance(x, str):
        s = x.strip().replace("\n", " ").strip("[]")
        return np.fromstring(s, sep=" ")

    raise TypeError(f"Unsupported embedding type: {type(x)}")


def embedding_features(campaign_emb, profile_emb) -> dict:
    c = to_vec(campaign_emb)
    p = to_vec(profile_emb)

    if c.shape != p.shape:
        raise ValueError(f"Different embedding shapes: {c.shape} vs {p.shape}")

    c_norm = np.linalg.norm(c)
    p_norm = np.linalg.norm(p)

    dot = float(np.dot(c, p))
    cosine = dot / (c_norm * p_norm + 1e-12)

    diff = c - p
    abs_diff = np.abs(diff)
    prod = c * p

    return {
        "emb_cosine": cosine,
        "emb_cosine_distance": 1 - cosine,
        "emb_dot": dot,
        "emb_euclidean": float(np.linalg.norm(diff)),
        "emb_manhattan": float(abs_diff.sum()),
        "emb_chebyshev": float(abs_diff.max()),
        "emb_campaign_norm": float(c_norm),
        "emb_profile_norm": float(p_norm),
        "emb_norm_ratio": float(c_norm / (p_norm + 1e-12)),
        "emb_absdiff_mean": float(abs_diff.mean()),
        "emb_absdiff_std": float(abs_diff.std()),
        "emb_absdiff_max": float(abs_diff.max()),
        "emb_prod_mean": float(prod.mean()),
        "emb_prod_std": float(prod.std()),
        "emb_prod_sum": float(prod.sum()),
    }


emb_features_df = pd.DataFrame([
    embedding_features(c_emb, p_emb)
    for c_emb, p_emb in zip(pairs["campaign_embedding"], pairs["profile_embedding"])
])

pairs = pd.concat(
    [pairs.reset_index(drop=True), emb_features_df],
    axis=1
)

pairs.shape

(301, 142)

In [21]:
def choose_validation_campaigns(
    df,
    campaign_col="campaign_original_index",
    target_col="ai_score",
    n_val_campaigns=7,
    must_val=252,
    must_train=246,
):
    all_campaigns = sorted(df[campaign_col].unique())

    assert must_val in all_campaigns, f"{must_val} нет в списке кампаний"
    assert must_train in all_campaigns, f"{must_train} нет в списке кампаний"

    campaign_mean = (
        df
        .groupby(campaign_col)[target_col]
        .mean()
        .drop(index=[must_val, must_train], errors="ignore")
        .sort_values()
    )

    candidates = campaign_mean.index.to_list()

    need = n_val_campaigns - 1

    raw_positions = np.linspace(0, len(candidates) - 1, need)
    positions = np.round(raw_positions).astype(int)

    picked = []
    used = set()

    for pos in positions:
        for radius in range(len(candidates)):
            candidate_positions = [pos - radius, pos + radius]

            for idx in candidate_positions:
                if 0 <= idx < len(candidates):
                    c = candidates[idx]
                    if c not in used:
                        picked.append(c)
                        used.add(c)
                        break

            if len(picked) == len(used):
                break

    val_campaigns = sorted([must_val] + picked[:need])

    return val_campaigns


VAL_CAMPAIGNS = choose_validation_campaigns(
    pairs,
    n_val_campaigns=7,
    must_val=252,
    must_train=246,
)

val_mask = pairs["campaign_original_index"].isin(VAL_CAMPAIGNS)

train_pairs = pairs.loc[~val_mask].copy()
val_pairs = pairs.loc[val_mask].copy()

assert 252 in set(val_pairs["campaign_original_index"])
assert 246 in set(train_pairs["campaign_original_index"])
assert 246 not in set(val_pairs["campaign_original_index"])
assert len(VAL_CAMPAIGNS) == 7
assert len(val_pairs) == 49

print("VAL_CAMPAIGNS:", VAL_CAMPAIGNS)
print("train rows:", len(train_pairs))
print("val rows:", len(val_pairs))

VAL_CAMPAIGNS: [66, 78, 84, 132, 168, 240, 252]
train rows: 252
val rows: 49


In [22]:
target_cols = [
    "ai_score",
    "target_prob",
]

drop_from_features = set([
    # target
    "ai_score",
    "target_prob",

    # raw embeddings
    "campaign_embedding",
    "profile_embedding",

    # identifiers / links / paths
    "campaign_original_index",
    "blogger_folder_number",
    "blogger_profile_id",
    "profile_id",
    "json_profile_id",
    "blogger_username",
    "json_username",
    "blogger_full_name",
    "json_full_name",
    "blogger_profile_dir",
    "blogger_analysis_path",
    "json_path",
    "link",
    "json_profile_url",
    "json_external_url",

    # long raw text
    "campaign_brief",
    "campaign_text",
    "blogger_rendered_text",
    "json_bio",
])

# Можно оставить категориальные короткие признаки для CatBoost
candidate_feature_cols = [
    c for c in pairs.columns
    if c not in drop_from_features
]

# Убираем совсем бесполезные object-колонки, кроме выбранных категориальных
cat_features = [
    c for c in [
        "blogger_gender",
        "campaign_product",
        "campaign_video_format",
        "campaign_tov",
        "json_business_category",
    ]
    if c in candidate_feature_cols
]

numeric_feature_cols = []

for c in candidate_feature_cols:
    if c in cat_features:
        continue

    if pd.api.types.is_numeric_dtype(pairs[c]):
        numeric_feature_cols.append(c)

feature_cols = numeric_feature_cols + cat_features

print("n numeric:", len(numeric_feature_cols))
print("n categorical:", len(cat_features))
print("n total:", len(feature_cols))

feature_cols

n numeric: 87
n categorical: 3
n total: 90


['rank',
 'score',
 'blogger_profile_id_x',
 'blogger_followers_x',
 'blogger_followers_y',
 'inst_link_found',
 'json_found',
 'json_followers',
 'json_following',
 'json_posts_count',
 'json_verified',
 'json_private',
 'json_business_account',
 'videos_count',
 'avg_views',
 'avg_likes',
 'avg_comments',
 'ERR',
 'avg_ERR',
 'ER_followers',
 'posting_span_days',
 'days_since_last_post',
 'posts_per_30d_in_sample',
 'downloaded_share',
 'avg_caption_len',
 'avg_hashtags',
 'avg_mentions',
 'likes_sum',
 'likes_mean',
 'likes_median',
 'likes_max',
 'likes_min',
 'likes_std',
 'comments_sum',
 'comments_mean',
 'comments_median',
 'comments_max',
 'comments_min',
 'comments_std',
 'views_sum',
 'views_mean',
 'views_median',
 'views_max',
 'views_min',
 'views_std',
 'caption_len_sum',
 'caption_len_mean',
 'caption_len_median',
 'caption_len_max',
 'caption_len_min',
 'caption_len_std',
 'hashtags_sum',
 'hashtags_mean',
 'hashtags_median',
 'hashtags_max',
 'hashtags_min',
 'hashtag

In [23]:
X_train = train_pairs[feature_cols].copy()
X_val = val_pairs[feature_cols].copy()

y_train = train_pairs["target_prob"].astype(float)
y_val = val_pairs["target_prob"].astype(float)

# Категориальные признаки CatBoost хочет строками
for col in cat_features:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col] = X_val[col].fillna("missing").astype(str)

# Числовые признаки приводим к numeric
for col in numeric_feature_cols:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_val[col] = pd.to_numeric(X_val[col], errors="coerce")

cat_feature_indices = [
    X_train.columns.get_loc(c)
    for c in cat_features
]

train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_feature_indices,
    feature_names=feature_cols,
)

val_pool = Pool(
    data=X_val,
    label=y_val,
    cat_features=cat_feature_indices,
    feature_names=feature_cols,
)

In [37]:
cat_model = CatBoostClassifier(
    loss_function="CrossEntropy",
    eval_metric="CrossEntropy",

    iterations=3000,
    learning_rate=0.03,
    depth=3,

    l2_leaf_reg=10,
    random_strength=1.0,
    bagging_temperature=1.0,

    random_seed=42,
    verbose=100,
    allow_writing_files=False,
)

cat_model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True,
    early_stopping_rounds=150,
)

0:	learn: 0.6856166	test: 0.6829843	best: 0.6829843 (0)	total: 21.7ms	remaining: 1m 5s
100:	learn: 0.5734512	test: 0.5265110	best: 0.5265110 (100)	total: 2.96s	remaining: 1m 24s
200:	learn: 0.5660326	test: 0.5173667	best: 0.5173581 (199)	total: 6.1s	remaining: 1m 24s
300:	learn: 0.5606007	test: 0.5145017	best: 0.5144126 (287)	total: 9.11s	remaining: 1m 21s
400:	learn: 0.5561950	test: 0.5117124	best: 0.5116852 (398)	total: 12.3s	remaining: 1m 19s
500:	learn: 0.5534639	test: 0.5102513	best: 0.5102513 (500)	total: 15.5s	remaining: 1m 17s
600:	learn: 0.5515310	test: 0.5088620	best: 0.5088620 (600)	total: 18.6s	remaining: 1m 14s
700:	learn: 0.5498059	test: 0.5076281	best: 0.5075397 (693)	total: 21.6s	remaining: 1m 10s
800:	learn: 0.5487087	test: 0.5069316	best: 0.5069198 (795)	total: 24.7s	remaining: 1m 7s
900:	learn: 0.5476006	test: 0.5065472	best: 0.5065172 (870)	total: 27.7s	remaining: 1m 4s
1000:	learn: 0.5468041	test: 0.5062618	best: 0.5062618 (1000)	total: 30.8s	remaining: 1m 1s
1100:

CatBoostClassifier(allow_writing_files=False, bagging_temperature=1.0, depth=3, eval_metric='CrossEntropy', iterations=3000, l2_leaf_reg=10, learning_rate=0.03, loss_function='CrossEntropy', random_seed=42, random_strength=1.0, verbose=100)

In [ ]:
cat_model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True,
    early_stopping_rounds=80,
)

0:	learn: 0.6885283	test: 0.6858256	best: 0.6858256 (0)	total: 1.79ms	remaining: 2.15s
100:	learn: 0.5962457	test: 0.5500882	best: 0.5500882 (100)	total: 2.08s	remaining: 22.7s
200:	learn: 0.5867701	test: 0.5356569	best: 0.5356546 (198)	total: 4.16s	remaining: 20.7s
300:	learn: 0.5823142	test: 0.5295676	best: 0.5295676 (300)	total: 6.23s	remaining: 18.6s
400:	learn: 0.5795270	test: 0.5261527	best: 0.5261511 (398)	total: 8.26s	remaining: 16.5s
500:	learn: 0.5778809	test: 0.5248115	best: 0.5248115 (500)	total: 10.4s	remaining: 14.5s
600:	learn: 0.5760366	test: 0.5236482	best: 0.5236325 (598)	total: 12.5s	remaining: 12.4s
700:	learn: 0.5748225	test: 0.5228365	best: 0.5228365 (700)	total: 14.6s	remaining: 10.4s
800:	learn: 0.5737355	test: 0.5224504	best: 0.5223757 (779)	total: 16.7s	remaining: 8.31s
900:	learn: 0.5726583	test: 0.5216491	best: 0.5216371 (897)	total: 18.8s	remaining: 6.22s
1000:	learn: 0.5717519	test: 0.5214391	best: 0.5214174 (997)	total: 20.9s	remaining: 4.14s
1100:	learn:

CatBoostClassifier(allow_writing_files=False, bagging_temperature=6.0, bootstrap_type='Bayesian', border_count=32, depth=2, eval_metric='CrossEntropy', iterations=1200, l2_leaf_reg=50, learning_rate=0.025, loss_function='CrossEntropy', model_size_reg=2.0, random_seed=42, random_strength=8.0, rsm=0.75, verbose=100)

In [38]:
train_pred_prob = cat_model.predict_proba(train_pool)[:, 1]
val_pred_prob = cat_model.predict_proba(val_pool)[:, 1]

train_pred_ai_score = np.clip(train_pred_prob * 100, 0, 100)
val_pred_ai_score = np.clip(val_pred_prob * 100, 0, 100)

In [39]:
def campaign_ndcg(g):
    return ndcg_score(
        [g["ai_score"].values],
        [g["pred_ai_score"].values],
    )


def make_eval_df(pairs_df, pred_prob, pred_ai_score):
    cols = [
        "campaign_original_index",
        "blogger_folder_number",
        "blogger_username",
        "blogger_full_name",
        "link",
        "ai_score",
    ]

    optional_cols = [
        "campaign_product",
        "campaign_video_format",
        "campaign_tov",
        "score",
        "rank",
        "avg_views",
        "avg_likes",
        "avg_comments",
        "ERR",
        "avg_ERR",
        "ER_followers",
    ]

    cols += [c for c in optional_cols if c in pairs_df.columns]
    cols = [c for c in cols if c in pairs_df.columns]

    out = pairs_df[cols].copy()
    out["pred_prob"] = pred_prob
    out["pred_ai_score"] = pred_ai_score
    out["abs_error"] = np.abs(out["ai_score"] - out["pred_ai_score"])

    out = out.sort_values(
        ["campaign_original_index", "pred_ai_score"],
        ascending=[True, False],
    )

    return out


train_eval = make_eval_df(train_pairs, train_pred_prob, train_pred_ai_score)
val_eval = make_eval_df(val_pairs, val_pred_prob, val_pred_ai_score)


def calc_ndcg_by_campaign(eval_df, split_name):
    ndcg_by_campaign = (
        eval_df
        .groupby("campaign_original_index")
        .apply(campaign_ndcg)
        .rename("ndcg")
        .reset_index()
    )

    mean_ndcg = ndcg_by_campaign["ndcg"].mean()

    print(f"{split_name} Mean NDCG: {mean_ndcg:.4f}")

    return ndcg_by_campaign, mean_ndcg


train_ndcg_by_campaign, train_mean_ndcg = calc_ndcg_by_campaign(train_eval, "Train")
val_ndcg_by_campaign, val_mean_ndcg = calc_ndcg_by_campaign(val_eval, "Validation")

Train Mean NDCG: 0.9949
Validation Mean NDCG: 0.8996


In [36]:
ndcg_summary = pd.DataFrame([
    {
        "split": "train",
        "mean_ndcg": train_mean_ndcg,
        "n_campaigns": train_eval["campaign_original_index"].nunique(),
        "n_rows": len(train_eval),
    },
    {
        "split": "validation",
        "mean_ndcg": val_mean_ndcg,
        "n_campaigns": val_eval["campaign_original_index"].nunique(),
        "n_rows": len(val_eval),
    },
])

ndcg_summary

,split,mean_ndcg,n_campaigns,n_rows
0,train,0.979348,36,252
1,validation,0.899475,7,49


In [40]:
CATBOOST_V2_MODEL_PATH = "catboost_soft_labels.cbm"
CATBOOST_V2_ARTIFACT_PATH = "catboost_soft_labels_v2_regularized_artifact.pkl"

cat_model.save_model(CATBOOST_V2_MODEL_PATH)